In [ ]:
from meteofetch import AromeOutreMerIndien
from aropangue.settings import DATA_DIR, PARAGLIDING_SITES, REUNION_BBOX
from aropangue.utils import list_available_model_arch

### Retrieving data

In [ ]:

# SP1 package contains surface wind at 10m (u10, v10) among other fields.
# Restricting to u10/v10 keeps memory usage low.
print("Getting Surface AROME data")
datasets = AromeOutreMerIndien.get_latest_forecast(
    paquet="SP2",
    # variables=("mx2t", "sh2"),
    path=DATA_DIR
)


In [ ]:
print("Getting 3D AROME data") 
datasets_3d = AromeOutreMerIndien.get_latest_forecast(
    paquet="HP1",
    path=DATA_DIR
)

Converting full grib2 to netcdf limited to Reunion island domain

In [ ]:
from aropangue.format import subset_grib_to_netcdf

grb_files = list_available_model_arch(arch_type='grib2')


for grb_p in grb_files.path :
    try :
        subset_grib_to_netcdf(
        grb_p, 
        bbox=REUNION_BBOX,
        model=AromeOutreMerIndien)
    except Exception as e:
        print(f'{grb_p} could not be susbetted : {e}')

### Loading available data

In [ ]:
from aropangue.dataloader import load_model_run
from datetime import datetime

PRINT_DS_INFO = False
run_date = datetime(2026,9,30,0)

ds_surf = load_model_run(
    "SP2",
    run_date,
    as_poi_subset=True,
    poi=PARAGLIDING_SITES
)

ds_3d = load_model_run(
    "HP1",
    run_date,
    as_poi_subset=True,
    poi=PARAGLIDING_SITES
)

elevations = ds_3d['t'].coords["heightAboveGround"].values

if PRINT_DS_INFO : 
    for ds, info in zip([ds_surf, ds_3d], ['surface', '3D']):
        print(info)
        for name, da in ds.data_vars.items():
            print(f"{name}:")
            print(f"  standard_name: {da.attrs.get('standard_name', 'N/A')}")
            # print(f"  dims: {da.dims}")
            # print(f"  shape: {da.shape}")
            print(f"  units: {da.attrs.get('units', 'N/A')}")
            # print(f"  attrs: {da.attrs}")


### Processing data

In [ ]:
from aropangue.basic_thermodynamics import (
    get_virtual_temp,
    extend_elevation_by_dry_adiab,
    add_specific_and_virtual_temperature, 
    get_lcl_height,
    get_ccl_height
)

print("compute diags")

ds_surf["Tv"] = get_virtual_temp(ds_surf["mx2t"], ds_surf["sh2"])
lcl = get_lcl_height(ds_surf)
lcl_mxt = get_lcl_height(ds_surf, temp_key='mx2t')
ccl = get_ccl_height(ds_3d)
particle_tv = extend_elevation_by_dry_adiab(
    ds_surf["Tv"], 
    elevations=elevations)

ds_3d = add_specific_and_virtual_temperature(ds_3d)
ds_3d["Tv"] =  ds_3d["Tv"].metpy.quantify()

In [ ]:
from aropangue.plots import plot_ceiling_height


plot_ceiling_height(lcl, pointname='COLILMACONS', ceiling_name='LCL')
plot_ceiling_height(lcl_mxt, pointname='COLILMACONS', ceiling_name='LCL')
plot_ceiling_height(ccl, pointname='COLILMACONS', ceiling_name='CCL')

In [ ]:
from aropangue.plots import plot_3d_clouds_hovmoller

rh = ds_3d["r"].squeeze().compute()
lcl = lcl.squeeze("point").compute()

plot_3d_clouds_hovmoller(rh, lcl)

In [ ]:
lcl

In [ ]:
from aropangue.plots import plot_tv_difference_hovmoller
from matplotlib import pyplot as plt

tv_diff = - (ds_3d["Tv"].compute() - particle_tv.compute())
u = ds_3d["u"]
v = ds_3d["v"]

point='Colimacons'
fig, ax = plot_tv_difference_hovmoller(
    tv_diff.sel(point=point), 
    u.sel(point=point), 
    v.sel(point=point), 
    add_windbarbs=True,
    pointname='Colimacons')
plt.show()

In [ ]:
lcl

In [ ]:
print(ds_3d["Tv"].time)
print(particle_tv.time)

print(ds_3d["Tv"].time.to_index().duplicated().sum())
print(particle_tv.time.to_index().duplicated().sum())

In [ ]:
print(particle_tv.metpy.units)
print(datasets_3d["Tv"].metpy.units)

In [ ]:
datasets_3d['Tv']

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs


fig, ax = plt.subplots(
    figsize=(10, 7),
    subplot_kw={"projection": ccrs.PlateCarree()},
)

(t-tv).isel(time=10).plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    x="longitude",
    y="latitude",
    cmap="RdBu_r",
)

ax.coastlines()
ax.set_title(f"w — {(t-tv).isel(time=10).time.values}")

plt.show()

In [ ]:
# # Wind speed and direction derived from the components
# wind_speed = np.sqrt(u10**2 + v10**2)
# wind_speed.name = "wind_speed_10m"

# print(u10)
# print(v10)
# print(wind_speed)

# # Example: extract the nearest grid point to Saint-Leu, Reunion
# lat_saint_leu = -21.17
# lon_saint_leu = 55.32

# point_speed = wind_speed.sel(
#     latitude=lat_saint_leu, longitude=lon_saint_leu, method="nearest"
# )
# print(point_speed)

In [ ]:
"""
Fetch the latest AROME 10m wind forecast (u10, v10) for the
Reunion Island / Indian Ocean domain using meteofetch.
"""

import metpy
import metpy.calc as mpcalc
import numpy as np
from meteofetch import AromeOutreMerIndien

# SP1 package contains surface wind at 10m (u10, v10) among other fields.
# Restricting to u10/v10 keeps memory usage low.
print("Getting AROME data")
datasets = AromeOutreMerIndien.get_latest_forecast(
    paquet="SP1",
    variables=("t2m", "r2", "prmsl"),
)

print("Quantifying units for metpy use")
t = datasets["t2m"].metpy.quantify()
rh = datasets["r2"].metpy.quantify()
p = datasets["prmsl"].metpy.quantify()

print("compute diags")
r = mpcalc.mixing_ratio_from_relative_humidity(p, t, rh)
Tv = mpcalc.virtual_temperature(t, rh)


In [ ]:
Tv